# Create / update the PolicySystem_Medallion job

Writes the full 11-task job definition into the existing job **569114070648966** (same Job ID, so ADF keeps
pointing at it). All tasks run on **serverless** (no cluster settings) in **Standard** performance mode.

```
bronze_branch   -> silver_branch   -> silver_agent  <- bronze_agent
bronze_customer -> silver_customer -> silver_policy <- bronze_policy
                   silver_policy   -> silver_claim  <- bronze_claim
                   silver_claim    -> gold
```

1. **Run all** with `apply = NO`: it only finds the notebooks and prints the plan.
2. Check every task has a path, then set `apply = YES` and **Run all** again.

Safe to re-run: it replaces the job definition with this one each time.

In [0]:
dbutils.widgets.dropdown("apply", "NO", ["NO", "YES"], "Write the job definition")
dbutils.widgets.dropdown("delete_old_jobs", "NO", ["NO", "YES"], "Delete the 4 old *-Ingestion jobs")

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()  # authenticates as you automatically inside a notebook

JOB_ID   = 569114070648966
JOB_NAME = "PolicySystem_Medallion"
HOME     = f"/Users/{w.current_user.me().user_name}"

# task_key -> (notebook name, [task_keys it depends on])
TASKS = {
    "bronze_branch":   ("BranchData_Ingestion",           []),
    "bronze_customer": ("CustomerData_Ingestion",         []),
    "bronze_agent":    ("AgentData_Ingestion",            []),
    "bronze_policy":   ("PolicyData_Ingestion",           []),
    "bronze_claim":    ("ClaimData_Ingestion",            []),
    "silver_branch":   ("Silver_Branch_Implementation",   ["bronze_branch"]),
    "silver_customer": ("Silver_Customer_Implementation", ["bronze_customer"]),
    "silver_agent":    ("Silver_Agent_Implementation",    ["bronze_agent", "silver_branch"]),
    "silver_policy":   ("Silver_Policy_Implementation",   ["bronze_policy", "silver_customer"]),
    "silver_claim":    ("Silver_Claim_Implementation",    ["bronze_claim", "silver_policy"]),
    "gold":            ("Gold_Layer_DataLoad",      ["silver_claim"]),
}

In [0]:
from databricks.sdk.service.workspace import ObjectType

found = {}
for obj in w.workspace.list(HOME, recursive=True):
    if obj.object_type == ObjectType.NOTEBOOK:
        found.setdefault(obj.path.rsplit("/", 1)[-1], []).append(obj.path)

paths, problems = {}, []
for key, (name, deps) in TASKS.items():
    matches = found.get(name, [])
    if len(matches) == 1:
        paths[key] = matches[0]
    elif not matches:
        problems.append(f"{key}: no notebook named '{name}' under {HOME}")
    else:
        problems.append(f"{key}: '{name}' exists {len(matches)} times -> rename or delete the extra copy: {matches}")

for key, (name, deps) in TASKS.items():
    print(f"{key:16s} depends on {', '.join(deps) or '-':32s} {paths.get(key, '*** NOT FOUND ***')}")

if problems:
    print("\nFix these before applying:")
    for p in problems:
        print(" -", p)

bronze_branch    depends on -                                /Users/kaushikchintalapudi@gmail.com/BronzeLayer Implementation/BranchData_Ingestion
bronze_customer  depends on -                                /Users/kaushikchintalapudi@gmail.com/BronzeLayer Implementation/CustomerData_Ingestion
bronze_agent     depends on -                                /Users/kaushikchintalapudi@gmail.com/BronzeLayer Implementation/AgentData_Ingestion
bronze_policy    depends on -                                /Users/kaushikchintalapudi@gmail.com/BronzeLayer Implementation/PolicyData_Ingestion
bronze_claim     depends on -                                /Users/kaushikchintalapudi@gmail.com/BronzeLayer Implementation/ClaimData_Ingestion
silver_branch    depends on bronze_branch                    /Users/kaushikchintalapudi@gmail.com/SilverLayer Implementation/Silver_Branch_Implementation
silver_customer  depends on bronze_customer                  /Users/kaushikchintalapudi@gmail.com/SilverLayer Implem

In [0]:
if dbutils.widgets.get("apply") != "YES":
    dbutils.notebook.exit("Plan only. Check the paths above, then set apply = YES and Run all again.")
if problems:
    raise Exception("Some notebooks were not found - see the list above.")

new_settings = {
    "name": JOB_NAME,
    "performance_target": "STANDARD",   # cheaper serverless mode; slower start is fine for a batch job
    "max_concurrent_runs": 1,           # never two runs writing the same tables at once
    "email_notifications": {"on_failure": [w.current_user.me().user_name]},
    "tasks": [
        {
            "task_key": key,
            "notebook_task": {"notebook_path": paths[key], "source": "WORKSPACE"},
            "depends_on": [{"task_key": d} for d in deps],
            "timeout_seconds": 3600,
            # no cluster settings -> serverless
        }
        for key, (name, deps) in TASKS.items()
    ],
    # no schedule: ADF triggers this job
}

w.api_client.do("POST", "/api/2.2/jobs/reset", body={"job_id": JOB_ID, "new_settings": new_settings})
print(f"Job {JOB_ID} updated with {len(new_settings['tasks'])} tasks.")
print(f"Open it: {w.config.host}/jobs/{JOB_ID}")

Job 569114070648966 updated with 11 tasks.
Open it: https://adb-7405612633267362.2.azuredatabricks.net/jobs/569114070648966


In [0]:
old = [j for j in w.jobs.list() if j.settings and j.settings.name.endswith("-Ingestion")]
for j in old:
    print(f"{j.job_id}  {j.settings.name}")

if dbutils.widgets.get("delete_old_jobs") == "YES":
    for j in old:
        w.jobs.delete(job_id=j.job_id)
        print("deleted", j.settings.name)
else:
    print("\nNot deleted. Set delete_old_jobs = YES to remove them (their notebooks are not touched).")

462013399874107  PolicyData-Ingestion
20992169384646  CustomerData-Ingestion
635516551309938  ClaimData-Ingestion
24174683553321  AgentData-Ingestion
327851663297348  BranchData-Ingestion

Not deleted. Set delete_old_jobs = YES to remove them (their notebooks are not touched).
